# Курсовий проєкт. Ноутбук 2 — згорткова мережа з нуля

Перший із двох підходів, які умова просить порівняти: мережа, навчена **з
нуля**, без жодних сторонніх ваг. Вона ж буде точкою відліку, відносно якої
вимірюється користь перенесення навчання в ноутбуці 3.

Десять прогонів, кожен відрізняється від базового **рівно одним чинником** —
інакше різницю в результаті не було б чому приписати:

| Прогін | Що змінюється |
|---|---|
| `a1_base` | базова Sequential-мережа без аугментації |
| `a2_aug` | аугментація |
| `a3_bn_gap` | Functional API, BatchNorm і GlobalAveragePooling |
| `a4`–`a5` | швидкість навчання 1e-2 / 1e-4 |
| `a6`–`a7` | розмір міні-вибірки 16 / 64 |
| `a8` | dropout 0.3 замість 0.5 |
| `a9` | сильна аугментація |
| `a10` | зважування класів |

Потрібен ГП. Час: 30–50 хвилин. Якщо сеанс обірветься — просто запустіть
ноутбук заново: виконані прогони зчитаються з Drive і пропустяться.

Середовище, шляхи, зерно. Якщо список GPU порожній — Середовище виконання → Змінити тип → ГП L4, і перезапустити клітинку.

In [ ]:
# Environment, paths, seed. Same prelude in every notebook of the project.
import json
import os
import shutil
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('TensorFlow', tf.__version__, '| Keras', keras.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
if IN_COLAB and not tf.config.list_physical_devices('GPU'):
    print('\n!! ГП НЕ УВІМКНЕНО. Середовище виконання -> Змінити тип -> ГП L4.')

PROJECT_ROOT = Path('/content') if IN_COLAB else Path.cwd().parent
RESULTS = PROJECT_ROOT / 'results'
(RESULTS / 'raw').mkdir(parents=True, exist_ok=True)

# Everything that must survive a Colab disconnect lives on Drive.
DRIVE_DIR = (Path('/content/drive/MyDrive/dl_final_project') if IN_COLAB
             else PROJECT_ROOT / 'data' / 'drive')


def mount_drive():
    """Mount Drive once; locally just ensure the stand-in folder exists."""
    if IN_COLAB and not Path('/content/drive/MyDrive').exists():
        from google.colab import drive

        drive.mount('/content/drive')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    return DRIVE_DIR


SEED = 42
keras.utils.set_random_seed(SEED)
plt.rcParams['figure.dpi'] = 110
print('Проєкт:', PROJECT_ROOT)
print('Drive :', DRIVE_DIR)

## 1. Підготовлений набір

Дерево з ноутбука 1: уже очищене, поділене й зведене до 320 px.

In [ ]:
# The prepared dataset comes from notebook 01: one tree with all 50 classes,
# already cleaned, split and resized to 320 px. Both tracks (top-15 and all-50)
# read from the same tree, so their splits are identical and comparable.
mount_drive()
PREPARED = DRIVE_DIR / 'prepared'
DATA_DIR = Path('/content/dataset') if IN_COLAB else PROJECT_ROOT / 'data' / 'dataset'

if not (DATA_DIR / 'train').exists():
    src = PREPARED / 'dataset.zip'
    if not src.exists():
        raise FileNotFoundError(
            f'немає {src} — спочатку виконайте ноутбук 01_dataset_eda.ipynb')
    t0 = time.time()
    with zipfile.ZipFile(src) as z:
        z.extractall(DATA_DIR)
    print(f'набір розпаковано за {time.time() - t0:.0f} с')

with open(PREPARED / 'class_names.json', encoding='utf-8') as f:
    CLASSES = json.load(f)
ALL50 = CLASSES['all50']
TOP15 = CLASSES['top15']

for _split in ('train', 'val', 'test'):
    _n = sum(1 for _ in (DATA_DIR / _split).rglob('*.jpg'))
    print(f'{_split:<6} {_n:>6} зображень')
print(f'\nкласів у наборі: {len(ALL50)}; основний трек: {len(TOP15)}')
print(', '.join(TOP15))

## 2. Конвеєр даних

Замість `ImageDataGenerator` з лабораторних тут `image_dataset_from_directory`
і `tf.data` — рідний спосіб Keras 3
([документація](https://keras.io/api/data_loading/image/)). Три переваги, які
тут справді потрібні:

1. `class_names` обмежує читання обраним треком, тож **обидві постановки
   (15 і 50 класів) живуть в одному дереві** без копіювання файлів;
2. `prefetch(AUTOTUNE)` готує наступну партію, поки ГП рахує поточну;
3. аугментація стає **шарами моделі**, а не параметрами генератора.

Останнє важливіше, ніж здається. Шари `Random*` активні лише під час навчання
й автоматично вимикаються на `predict` і `evaluate` — тобто вимогу «аугментація
лише до навчальної вибірки» неможливо порушити випадково. Виконуються вони на
ГП.

In [ ]:
# tf.data pipeline (Keras 3 way; the labs used the legacy ImageDataGenerator).
# `class_names` restricts the reader to the chosen track without copying files.
#
# Only val/test are cached in memory: the training set at 224x224 float32 would
# take several GB, and decoding 320 px JPEGs with AUTOTUNE is cheap enough.
AUTOTUNE = tf.data.AUTOTUNE

AUG_PRESETS = {
    'light': [
        keras.layers.RandomFlip('horizontal'),
        keras.layers.RandomRotation(0.05),
        keras.layers.RandomZoom(0.1),
    ],
    'strong': [
        keras.layers.RandomFlip('horizontal'),
        keras.layers.RandomRotation(0.15),
        keras.layers.RandomZoom(0.2),
        keras.layers.RandomTranslation(0.1, 0.1),
        keras.layers.RandomContrast(0.2),
    ],
}


def make_augment(kind):
    """Augmentation as model layers: runs on GPU and is inactive at inference.

    No vertical flip: a painting is never upside down, and mirroring already
    stretches what a brushstroke may look like.
    """
    if not kind:
        return None
    return keras.Sequential(AUG_PRESETS[kind], name=f'augment_{kind}')


def make_datasets(class_names, img_size=224, batch=32):
    def load(split, shuffle):
        return keras.utils.image_dataset_from_directory(
            DATA_DIR / split, labels='inferred', label_mode='int',
            class_names=class_names, image_size=(img_size, img_size),
            batch_size=batch, shuffle=shuffle, seed=SEED,
            interpolation='bilinear', verbose=False)

    train = load('train', True).prefetch(AUTOTUNE)
    val = load('val', False).cache().prefetch(AUTOTUNE)
    test = load('test', False).cache().prefetch(AUTOTUNE)
    return train, val, test


def class_weights_for(class_names):
    """Balanced weights: rare artists get a proportionally larger penalty."""
    counts = np.array([len(list((DATA_DIR / 'train' / c).glob('*.jpg')))
                       for c in class_names], dtype=float)
    weights = counts.sum() / (len(counts) * counts)
    return {i: float(w) for i, w in enumerate(weights)}


def count_steps(class_names, split='train'):
    return sum(len(list((DATA_DIR / split / c).glob('*.jpg'))) for c in class_names)


print('зображень у навчальній вибірці трека top-15:', count_steps(TOP15))

## 3. Стенд прогонів

Одна функція `run()` на всі експерименти. Вона ж веде **реєстр на Drive**:
кожен завершений прогін записується в JSON і при повторному запуску
пропускається. Відвалився Colab — втрачено один експеримент, а не серія.

Колбеки:

- `EarlyStopping(restore_best_weights=True)` — повертає ваги найкращої епохи,
  тож пізнє перенавчання не псує підсумкову оцінку;
- `ReduceLROnPlateau` — зменшує швидкість навчання вдвічі, коли втрати
  перестають падати;
- `ModelCheckpoint` — на локальний диск, не на Drive: запис через змонтований
  Drive на кожному покращенні коштує більше, ніж саме навчання;
- `CSVLogger` і власний `TimeHistory` для таблиць у звіті.

In [ ]:
# One function for every experiment, so that between any two runs exactly one
# factor differs. Each finished run is written to Drive and skipped on re-run,
# so a Colab disconnect costs one experiment instead of the whole series.
RUNS_DIR = DRIVE_DIR / 'runs'
MODELS_DIR = DRIVE_DIR / 'models'
CKPT_DIR = PROJECT_ROOT / 'ckpt'
for _d in (RUNS_DIR, MODELS_DIR, CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

FORCE_RERUN = False          # True -> ignore saved runs and train everything again
results = {}


class TimeHistory(keras.callbacks.Callback):
    """Wall-clock time of training; reported in the summary table."""

    def on_train_begin(self, logs=None):
        self._t0 = time.time()
        self.seconds = 0.0

    def on_train_end(self, logs=None):
        self.seconds = time.time() - self._t0


def make_callbacks(name, timer, patience=5, reduce_lr=True):
    # Checkpoints go to local disk: writing every improvement straight to Drive
    # is slow. The best model is copied to Drive once, after training.
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max',
                                      patience=patience, restore_best_weights=True,
                                      verbose=1),
        keras.callbacks.ModelCheckpoint(str(CKPT_DIR / f'{name}.keras'),
                                        monitor='val_accuracy', mode='max',
                                        save_best_only=True),
        keras.callbacks.CSVLogger(str(RESULTS / 'raw' / f'{name}_log.csv')),
        timer,
    ]
    # ReduceLROnPlateau and a LearningRateSchedule are mutually exclusive: the
    # callback assigns a new value to optimizer.learning_rate, and an optimizer
    # built with a schedule refuses that (TypeError, mid-training). A schedule
    # already governs the learning rate, so the callback is simply dropped.
    if reduce_lr:
        callbacks.insert(1, keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1))
    return callbacks


def top_k_accuracy(model, ds, k=3):
    hits = total = 0
    for xb, yb in ds:
        probs = model.predict(xb, verbose=0)
        topk = np.argsort(probs, axis=1)[:, -k:]
        hits += int(np.sum(topk == np.asarray(yb)[:, None]))
        total += len(yb)
    return hits / max(total, 1)


def run(cfg, build_fn):
    """Train one configuration and record everything the report needs."""
    name = cfg['name']
    path = RUNS_DIR / f'{name}.json'
    if path.exists() and not FORCE_RERUN:
        results[name] = json.loads(path.read_text(encoding='utf-8'))
        print(f"[{name}] уже виконано ранiше (val {results[name]['best_val_accuracy']:.4f})"
              ' — пропускаю')
        return results[name]

    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)

    classes = cfg.get('classes', TOP15)
    train, val, test = make_datasets(classes, cfg.get('img_size', 224),
                                     cfg.get('batch', 32))
    model = build_fn(cfg, classes)
    weights = class_weights_for(classes) if cfg.get('class_weight') else None

    timer = TimeHistory()
    print(f"\n{'=' * 70}\n{name}: {cfg['title']}\n{'=' * 70}")
    history = model.fit(train, validation_data=val, epochs=cfg.get('epochs', 25),
                        class_weight=weights, verbose=2,
                        callbacks=make_callbacks(name, timer,
                                                 cfg.get('patience', 5),
                                                 reduce_lr=not cfg.get('schedule')))

    test_loss, test_acc = model.evaluate(test, verbose=0)
    record = {
        'name': name,
        'title': cfg['title'],
        'config': {k: v for k, v in cfg.items() if k != 'classes'},
        'n_classes': len(classes),
        'params': int(model.count_params()),
        'trainable_params': int(sum(np.prod(w.shape) for w in model.trainable_weights)),
        'epochs_run': len(history.history['loss']),
        'best_val_accuracy': float(max(history.history['val_accuracy'])),
        'final_train_accuracy': float(history.history['accuracy'][-1]),
        'test_accuracy': float(test_acc),
        'test_loss': float(test_loss),
        'test_top3_accuracy': float(top_k_accuracy(model, test, 3)),
        'seconds': float(timer.seconds),
        'history': {k: [float(x) for x in v] for k, v in history.history.items()},
    }
    record['overfit_gap'] = record['final_train_accuracy'] - record['best_val_accuracy']

    model.save(MODELS_DIR / f'{name}.keras')
    path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding='utf-8')
    results[name] = record
    print(f"\n[{name}] val {record['best_val_accuracy']:.4f} | "
          f"test {record['test_accuracy']:.4f} | top-3 {record['test_top3_accuracy']:.4f} | "
          f"{record['seconds'] / 60:.1f} хв")

    del model
    keras.backend.clear_session()
    return record


def load_previous_runs():
    """Pick up runs made in an earlier session (or in an earlier notebook)."""
    for f in sorted(RUNS_DIR.glob('*.json')):
        results.setdefault(f.stem, json.loads(f.read_text(encoding='utf-8')))
    print(f'відомо прогонів: {len(results)}')


load_previous_runs()

In [ ]:
# Summary table of every run known so far. pandas is a deliberate departure
# from the labs' hand-built ASCII tables: these numbers go straight into the report.
def summary_frame(names=None):
    names = names or list(results)
    rows = []
    for name in names:
        r = results[name]
        rows.append({
            'експеримент': name,
            'опис': r['title'],
            'класів': r['n_classes'],
            'параметрів': r['params'],
            'навчаних': r['trainable_params'],
            'епох': r['epochs_run'],
            'val acc': round(r['best_val_accuracy'], 4),
            'test acc': round(r['test_accuracy'], 4),
            'test top-3': round(r['test_top3_accuracy'], 4),
            'розрив': round(r['overfit_gap'], 4),
            'хв': round(r['seconds'] / 60, 1),
        })
    df = pd.DataFrame(rows).sort_values('test acc', ascending=False)
    return df.reset_index(drop=True)

In [ ]:
# Learning curves. Solid red = training, dashed blue = validation;
# the distance between them is overfitting.
def plot_histories(names, fname, suptitle):
    names = [n for n in names if results.get(n, {}).get('history')]
    if not names:
        print('немає історій для графіків')
        return
    cols = min(3, len(names))
    rows = -(-len(names) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5.2 * cols, 3.9 * rows),
                             squeeze=False)
    for ax, name in zip(axes.ravel(), names):
        rec = results[name]
        h = rec['history']
        epochs = range(1, len(h['accuracy']) + 1)
        # Маркери, а не самі лінії: прогін, зупинений після однієї епохи, інакше
        # не було б видно взагалі.
        ax.plot(epochs, h['accuracy'], color='crimson', marker='o', ms=3,
                label='навчальна')
        ax.plot(epochs, h['val_accuracy'], color='royalblue', ls='--',
                marker='s', ms=3, label='перевірочна')
        ax.axhline(rec['test_accuracy'], color='seagreen', ls=':', lw=1.2,
                   label='тестова')
        ax.set_title(f"{name}\n{rec['title']}", fontsize=9)
        ax.set_xlabel('епоха')
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.set_ylabel('точність')
        ax.set_ylim(0, 1)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=7)
    for ax in axes.ravel()[len(names):]:
        ax.axis('off')
    fig.suptitle(suptitle, fontsize=13)
    plt.tight_layout()
    plt.savefig(RESULTS / fname, dpi=110, bbox_inches='tight')
    plt.show()

## 4. Дві архітектури

`build_sequential` — звичний стек Conv/Pool → Flatten → Dense зі Sequential API.

`build_functional` — та сама глибина, записана через Functional API, плюс дві
зміни: `BatchNormalization` після кожної згортки і `GlobalAveragePooling2D`
замість `Flatten`. Друга зміна прибирає з мережі більшу частину параметрів:
`Flatten` на карті 14×14×128 дає вектор на 25 тисяч чисел, і наступний
`Dense(512)` сам по собі важить 12.8 млн ваг — при тому, що навчальних
зображень лише кілька тисяч.

Нормалізація пікселів — шаром `Rescaling` **усередині** моделі, тому збережена
модель сама обробляє свій вхід, і в ноутбуці 4 її неможливо погодувати
неправильно.

In [ ]:
# Two architectures, both trained from scratch.
#
# `build_sequential` is the Sequential style of labs 7-8: Conv/Pool stack ->
# Flatten -> Dense. `build_functional` is the same depth written with the
# Functional API and two modern additions — BatchNorm after every convolution
# and GlobalAveragePooling instead of Flatten, which removes ~12M parameters
# that exist only to memorise the training set.
#
# Normalisation is a Rescaling layer inside the model, so a saved model
# preprocesses its own input and notebook 04 cannot get it wrong.
def make_optimizer(cfg):
    lr = cfg.get('lr', 1e-3)
    name = cfg.get('optimizer', 'adam')
    if name == 'adam':
        return keras.optimizers.Adam(learning_rate=lr)
    if name == 'rmsprop':
        return keras.optimizers.RMSprop(learning_rate=lr)
    if name == 'sgd':
        return keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    raise ValueError(f'невідомий оптимізатор: {name}')


def compile_model(model, cfg):
    model.compile(optimizer=make_optimizer(cfg),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model


def build_sequential(cfg, classes):
    size = cfg.get('img_size', 224)
    augment = make_augment(cfg.get('augment'))
    model = keras.Sequential(name=cfg['name'])
    model.add(keras.Input(shape=(size, size, 3)))
    if augment is not None:
        model.add(augment)
    model.add(keras.layers.Rescaling(1. / 255))
    for filters in cfg.get('filters', (32, 64, 128, 128)):
        model.add(keras.layers.Conv2D(filters, 3, padding='same', activation='relu'))
        model.add(keras.layers.MaxPooling2D(2))
    model.add(keras.layers.Flatten())
    model.add(keras.layers.Dropout(cfg.get('dropout', 0.5)))
    model.add(keras.layers.Dense(512, activation='relu'))
    model.add(keras.layers.Dense(len(classes), activation='softmax'))
    return compile_model(model, cfg)


def build_functional(cfg, classes):
    size = cfg.get('img_size', 224)
    augment = make_augment(cfg.get('augment'))
    inputs = keras.Input(shape=(size, size, 3))
    x = augment(inputs) if augment is not None else inputs
    x = keras.layers.Rescaling(1. / 255)(x)
    for filters in cfg.get('filters', (32, 64, 128, 128)):
        x = keras.layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
        x = keras.layers.BatchNormalization()(x)
        x = keras.layers.Activation('relu')(x)
        x = keras.layers.MaxPooling2D(2)(x)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(cfg.get('dropout', 0.5))(x)
    outputs = keras.layers.Dense(len(classes), activation='softmax')(x)
    model = keras.Model(inputs, outputs, name=cfg['name'])
    return compile_model(model, cfg)


BUILDERS = {'seq': build_sequential, 'func': build_functional}


def run_cfg(cfg):
    return run(cfg, BUILDERS[cfg['builder']])


_demo = build_functional({'name': 'demo', 'augment': 'light'}, TOP15)
_demo.summary()
del _demo
keras.backend.clear_session()

## 5. Архітектура й аугментація

In [ ]:
# A1-A3: architecture and augmentation. Each pair differs in exactly one thing.
run_cfg({'name': 'a1_base', 'builder': 'seq', 'augment': None,
         'title': 'Sequential CNN, без аугментації'})
run_cfg({'name': 'a2_aug', 'builder': 'seq', 'augment': 'light',
         'title': 'Sequential CNN + аугментація'})
run_cfg({'name': 'a3_bn_gap', 'builder': 'func', 'augment': 'light',
         'title': 'Functional API, BatchNorm + GlobalAveragePooling'})

print(summary_frame(['a1_base', 'a2_aug', 'a3_bn_gap']).to_string(index=False))

## 6. Підбір гіперпараметрів

Варіанти будуються не навмання, а навколо найкращої з трьох попередніх
конфігурацій, і кожен змінює рівно одне значення.

In [ ]:
# A4-A9: hyperparameters, all varied around whichever of A1-A3 came out best,
# so every row below differs from that baseline in exactly one setting.
BEST_A = max(['a1_base', 'a2_aug', 'a3_bn_gap'],
             key=lambda n: results[n]['best_val_accuracy'])
BASE_CFG = dict(results[BEST_A]['config'])
print(f'базова конфігурація для підбору: {BEST_A} — {results[BEST_A]["title"]}')


def variant(name, title, **overrides):
    cfg = {**BASE_CFG, **overrides}
    cfg['name'] = name
    cfg['title'] = title
    return run_cfg(cfg)


variant('a4_lr_1e2', 'швидкість навчання 1e-2', lr=1e-2)
variant('a5_lr_1e4', 'швидкість навчання 1e-4', lr=1e-4)
variant('a6_batch_16', 'міні-вибірка 16', batch=16)
variant('a7_batch_64', 'міні-вибірка 64', batch=64)
variant('a8_dropout_03', 'dropout 0.3 замість 0.5', dropout=0.3)
variant('a9_aug_strong', 'сильна аугментація', augment='strong')
variant('a10_class_weight', 'зважування класів', class_weight=True)

## 7. Результати

In [ ]:
# What the ten runs say. Deltas are computed, not typed in by hand.
SCRATCH_RUNS = [n for n in results if n.startswith(('a1', 'a2', 'a3', 'a4', 'a5',
                                                    'a6', 'a7', 'a8', 'a9', 'a10'))]
table = summary_frame(SCRATCH_RUNS)
print(table.to_string(index=False))

BEST_SCRATCH = table.iloc[0]['експеримент']
print(f"\nнайкраща мережа з нуля: {BEST_SCRATCH} — "
      f"test {results[BEST_SCRATCH]['test_accuracy']:.4f}, "
      f"top-3 {results[BEST_SCRATCH]['test_top3_accuracy']:.4f}")


def delta(a, b, field='test_accuracy'):
    if a not in results or b not in results:
        return
    va, vb = results[a][field], results[b][field]
    print(f'  {a} -> {b}: {va:.4f} -> {vb:.4f} ({vb - va:+.4f}); '
          f"розрив {results[a]['overfit_gap']:+.3f} -> {results[b]['overfit_gap']:+.3f}")


print('\nвплив окремих чинників (тестова точність і розрив train-val):')
delta('a1_base', 'a2_aug')
delta('a2_aug', 'a3_bn_gap')
delta(BEST_A, 'a4_lr_1e2')
delta(BEST_A, 'a5_lr_1e4')
delta(BEST_A, 'a6_batch_16')
delta(BEST_A, 'a7_batch_64')
delta(BEST_A, 'a8_dropout_03')
delta(BEST_A, 'a9_aug_strong')
delta(BEST_A, 'a10_class_weight')

table.to_csv(RESULTS / 'raw' / 'nb02_table.csv', index=False, encoding='utf-8')
print('\nтаблицю збережено у results/raw/nb02_table.csv')

Криві навчання. Відстань між суцільною червоною (навчальна) і
пунктирною синьою (перевірочна) — це перенавчання; зелена пунктирна — точність
на тестовій вибірці.

In [ ]:
plot_histories(SCRATCH_RUNS, 'nb02_histories.png',
                'Криві навчання: згорткові мережі, навчені з нуля')

## 8. Експорт

Зіп покласти у `project/results/incoming/`, ноутбук — через
**Файл → Завантажити → Завантажити .ipynb** у `project/notebooks/`.

In [ ]:
# Export: numbers for the report plus a zip to drop into results/incoming/.
NB = 'nb02'
payload = {
    'notebook': NB,
    'tensorflow': tf.__version__,
    'gpu': [d.name for d in tf.config.list_physical_devices('GPU')],
    'runs': results,
}
out = RESULTS / 'raw' / f'{NB}.json'
out.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('записано', out)

zip_path = PROJECT_ROOT / f'project_{NB}_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(RESULTS.rglob('*')):
        if f.is_file() and f.suffix in ('.json', '.png', '.csv', '.md'):
            z.write(f, f.relative_to(RESULTS))
print('зіп:', zip_path, f'({zip_path.stat().st_size / 1e6:.1f} МБ)')

if IN_COLAB:
    from google.colab import files

    files.download(str(zip_path))